# Limit LSS covariance matrix check
The GPY test requires to compute the covariance matrix of the LSSs in the limit where N,M tends to infinity at some specified rate M/N=c.

The computation of this matrix is not trivial, so this notebook is used as an additional check. 

# Empirical covariance

In [1]:
# check that the sample test statistics follow the same covariance
from gpy_test import GPY
from gpy_test.result import GPYResult

from statsmodels.tsa.arima_process import arma_generate_sample
import numpy as np
from joblib import Parallel, delayed
from tqdm import tqdm


# the complex gaussian time series will be ARMA(1,1) with the following parameters
c = 0.2
ar = 0.1
ma = -0.1
N = 1000  # number of samples
M = int(c * N)  # number of time series


# for the GPY test, we will use the following test functions
fs = [lambda x: x, lambda x: x**2]


def run() -> GPYResult:
    real = arma_generate_sample([1, -ar], [1, ma], (N, M), scale=1 / np.sqrt(2))
    imag = arma_generate_sample([1, -ar], [1, ma], (N, M), scale=1 / np.sqrt(2))
    y = real + 1j * imag
    Cov = np.identity(len(fs))  # give a dummy covariance matrix, it won't be used
    return GPY(y, fs, Cov)


# Generate a large number of experiments
n_repeat = 100
results = Parallel(n_jobs=-1)(delayed(run)() for _ in tqdm(range(n_repeat)))

# show the empirical covariance of the test statistics
np.cov(np.array([result.lss for result in results]).T)

100%|██████████| 10/10 [00:04<00:00,  2.50it/s]


array([[ 1.02171592,  3.13702328],
       [ 3.13702328, 10.09004773]])

# Limit Covariance

In [2]:
from ruamel.yaml import YAML
from pathlib import Path
from gpy_test.config.covariance import CovarianceConfig
import os
from gpy_test.covariance import covariance

# load the base config
current_path = os.path.abspath("")
current_dir = Path(current_path).parent
yaml = YAML(typ="safe")
with open(current_dir.parent / "src" / "gpy_test" / "config.yaml") as f:
    config = yaml.load(f)
base_covariance_config = CovarianceConfig(**config)

# we also need to know the range of the eigenvalues of the covariance matrix. For that we just use
# the range of the eigenvalues of the sample covariance matrix for one of the experiments
eig_range = np.min(results[0].eigs_S_1), np.max(results[0].eigs_S_1)


# we will also need to provide the spectral density of the ARMA(1,1) process. It can also be computed
# from the sample time series.
def _ARMA_spectral_density(ar: float, ma: float) -> callable:
    ma_part = lambda nu: 1 + ma**2 + 2 * ma * np.cos(2 * np.pi * nu)
    ar_part = lambda nu: 1 / (1 + ar**2 - 2 * ar * np.cos(2 * np.pi * nu))
    return lambda nu: ar_part(nu) * ma_part(nu)


oracle_sd = _ARMA_spectral_density(ar, ma)

In [14]:
d = base_covariance_config.dict()
d["integral_config"]["type_"] = "dblsimpson"
d["integral_config"]["n_points"] = 250
d["contour_config_pair"][0]["type_"] = "ellipse"
d["contour_config_pair"][1]["type_"] = "ellipse"
covariance_config = CovarianceConfig(**d)

covariance(
    covariance_config,
    fs,
    oracle_sd,
    eig_range,
    c,
)

100%|██████████| 3/3 [00:00<00:00, 997.06it/s]


array([[0.8000802 , 1.9203702 ],
       [1.9203702 , 4.92975596]])

In [15]:
d = covariance_config.dict()
d["type_"] = "dblsimpson"
d["n_points"] = 250
d["contour_config_pair"][0]["type_"] = "rectangle"
d["contour_config_pair"][1]["type_"] = "rectangle"
covariance_config = CovarianceConfig(**d)

covariance(
    covariance_config,
    fs,
    oracle_sd,
    eig_range,
    c,
)

100%|██████████| 3/3 [00:00<00:00, 1097.03it/s]


ValueError: Convergence issue: one entry of the limit covariance matrix is not real: omega_ij=(0.8011233259148932+0.014140200333997207j)

In [ ]:
d = covariance_config.dict()
d["type_"] = "dblsimpson"
d["n_points"] = 250
d["contour_config_pair"][0]["type_"] = "circle"
d["contour_config_pair"][1]["type_"] = "circle"
covariance_config = CovarianceConfig(**d)

covariance(
    covariance_config,
    fs,
    oracle_sd,
    eig_range,
    c,
)

100%|██████████| 3/3 [00:00<00:00, 1718.27it/s]


array([[0.80057789, 1.9202206 ],
       [1.9202206 , 4.92637484]])

In [ ]:
d = covariance_config.dict()
d["type_"] = "dblquad"
d["n_points"] = None
d["contour_config_pair"][0]["type_"] = "ellipse"
d["contour_config_pair"][1]["type_"] = "ellipse"
covariance_config = CovarianceConfig(**d)

covariance(
    covariance_config,
    fs,
    oracle_sd,
    eig_range,
    c,
)

100%|██████████| 3/3 [00:00<00:00, 892.09it/s]


array([[0.80057789, 1.9202206 ],
       [1.9202206 , 4.92637484]])

In [ ]:
d = covariance_config.dict()
d["type_"] = "dblquad"
d["n_points"] = None
d["contour_config_pair"][0]["type_"] = "rectangle"
d["contour_config_pair"][1]["type_"] = "rectangle"
covariance_config = CovarianceConfig(**d)

covariance(
    covariance_config,
    fs,
    oracle_sd,
    eig_range,
    c,
)

100%|██████████| 3/3 [00:00<00:00, 1940.61it/s]


array([[0.80057789, 1.9202206 ],
       [1.9202206 , 4.92637484]])

In [ ]:
d = covariance_config.dict()
d["type_"] = "dblquad"
d["n_points"] = None
d["contour_config_pair"][0]["type_"] = "circle"
d["contour_config_pair"][1]["type_"] = "circle"
covariance_config = CovarianceConfig(**d)

covariance(
    covariance_config,
    fs,
    oracle_sd,
    eig_range,
    c,
)

100%|██████████| 3/3 [00:00<00:00, 441.96it/s]


array([[0.80057789, 1.9202206 ],
       [1.9202206 , 4.92637484]])

In [ ]:
# try with samples

d = base_covariance_config.dict()
d["integral_config"]["type_"] = "dblsimpson"
d["integral_config"]["n_points"] = 250
d["contour_config_pair"][0]["type_"] = "ellipse"
d["contour_config_pair"][1]["type_"] = "ellipse"
covariance_config = CovarianceConfig(**d)

nus = np.linspace(-0.5, 0.5, 50)
sd_values = np.array([oracle_sd(nu) for nu in nus])

covariance(
    covariance_config,
    fs,
    (nus, sd_values),
    eig_range,
    c,
)

100%|██████████| 3/3 [00:00<00:00, 1454.84it/s]


array([[0.8000802 , 1.9203702 ],
       [1.9203702 , 4.92975598]])

In [ ]:
d = covariance_config.dict()
d["type_"] = "dblsimpson"
d["n_points"] = 250
d["contour_config_pair"][0]["type_"] = "rectangle"
d["contour_config_pair"][1]["type_"] = "rectangle"
covariance_config = CovarianceConfig(**d)

nus = np.linspace(-0.5, 0.5, 50)
sd_values = np.array([oracle_sd(nu) for nu in nus])

covariance(
    covariance_config,
    fs,
    oracle_sd,
    eig_range,
    c,
)

100%|██████████| 3/3 [00:00<00:00, 1834.78it/s]


array([[0.8000802 , 1.9203702 ],
       [1.9203702 , 4.92975596]])

In [ ]:
d = covariance_config.dict()
d["type_"] = "dblsimpson"
d["n_points"] = 250
d["contour_config_pair"][0]["type_"] = "circle"
d["contour_config_pair"][1]["type_"] = "circle"
covariance_config = CovarianceConfig(**d)

nus = np.linspace(-0.5, 0.5, 50)
sd_values = np.array([oracle_sd(nu) for nu in nus])

covariance(
    covariance_config,
    fs,
    (nus, sd_values),
    eig_range,
    c,
)

100%|██████████| 3/3 [00:00<00:00, 498.23it/s]


array([[0.8000802 , 1.9203702 ],
       [1.9203702 , 4.92975598]])

In [ ]:
d = covariance_config.dict()
d["type_"] = "dblquad"
d["n_points"] = None
d["contour_config_pair"][0]["type_"] = "ellipse"
d["contour_config_pair"][1]["type_"] = "ellipse"
covariance_config = CovarianceConfig(**d)

nus = np.linspace(-0.5, 0.5, 50)
sd_values = np.array([oracle_sd(nu) for nu in nus])

covariance(
    covariance_config,
    fs,
    (nus, sd_values),
    eig_range,
    c,
)

100%|██████████| 3/3 [00:00<00:00, 1209.20it/s]

array([[0.8000802 , 1.9203702 ],
       [1.9203702 , 4.92975598]])

In [ ]:
d = covariance_config.dict()
d["type_"] = "dblquad"
d["n_points"] = None
d["contour_config_pair"][0]["type_"] = "rectangle"
d["contour_config_pair"][1]["type_"] = "rectangle"
covariance_config = CovarianceConfig(**d)

nus = np.linspace(-0.5, 0.5, 50)
sd_values = np.array([oracle_sd(nu) for nu in nus])

covariance(
    covariance_config,
    fs,
    (nus, sd_values),
    eig_range,
    c,
)

100%|██████████| 3/3 [00:00<00:00, 3005.23it/s]


array([[0.8000802 , 1.9203702 ],
       [1.9203702 , 4.92975598]])

In [ ]:
d = covariance_config.dict()
d["type_"] = "dblquad"
d["n_points"] = None
d["contour_config_pair"][0]["type_"] = "circle"
d["contour_config_pair"][1]["type_"] = "circle"
covariance_config = CovarianceConfig(**d)

nus = np.linspace(-0.5, 0.5, 50)
sd_values = np.array([oracle_sd(nu) for nu in nus])

covariance(
    covariance_config,
    fs,
    (nus, sd_values),
    eig_range,
    c,
)

100%|██████████| 3/3 [00:00<00:00, 457.86it/s]


array([[0.8000802 , 1.9203702 ],
       [1.9203702 , 4.92975598]])

In [ ]:
# compute the spectral density of the time series if not provided. The GPY assumes that the time series
# share the same spectral density, so we average the spectral density of each time series.
if sd is None:
    frequency_grid = fftfreq(N)
    sd = lag_window(y, L=5)
    sd_values = [np.mean(sd(nu)) for nu in frequency_grid]
    # (frequency_grid, sd_values) = periodogram(y.T, scaling="density")
    # sd_values = np.mean(sd_values, axis=0)
else:
    frequency_grid = fftfreq(N)
    sd_values = np.array([sd(f) for f in frequency_grid])
    # sd = lag_window(y, L=gpy_config.L)

# make sure the frequency grid is increasing
frequency_grid = np.fft.fftshift(frequency_grid)
sd_values = np.fft.fftshift(sd_values, axes=0)